# Stockage des données sur Onyxia

Les données sont stockées sur le compte de "tomasmossel", et pour faire tourner ce notebook, il faudrait adapter le ID d'Utilisateur.


In [ ]:
!pip install -r requirements.txt

import numpy as np
import pandas as pd
import requests
import s3fs
import io
import zipfile
from io import BytesIO

from CODE.functions_emissions import extract_values
np.random.seed(123)

## Données sur les Emissions 

In [ ]:
# Données du Geoserver de Airparif 

url_base = "https://namek.airparif.fr/geoserver/Moyenne_annuelle/ows?service=WMS&version=1.3.0&request=GetFeatureInfo"
years = range(2007,2025)
particles = {"no2","o3","pm25"}


In [ ]:
# Données des centroïdes des communes de l'île de france

url_communes = "https://geo.api.gouv.fr/communes"
departement_idf = ["75","77","78","91","92","93","94","95"]

data_centroides = []

for dep in departement_idf:
    parametres = {"codeDepartement": dep, "type": "commune-actuelle,arrondissement-municipal","fields": "nom,code,codeDepartement,centre", "format": "json"}
    r = requests.get(url_communes, params = parametres).json()
    data_centroides += r
    
ile_de_fr_commune = pd.DataFrame({
    "INSEE_Commune" : [com["code"] for com in data_centroides],
    "INSEE_Departement" : [com["codeDepartement"] for com in data_centroides],
    "Nom_Commune" : [com["nom"] for com in data_centroides],
    "Latitude" : [com["centre"]["coordinates"][1] for com in data_centroides],
    "Longitude" : [com["centre"]["coordinates"][0] for com in data_centroides]})

ile_de_fr_commune = ile_de_fr_commune.drop(index=0).reset_index(drop=True) # la premiere ligne est Paris en general, donc on l'enleve

In [ ]:
# Scraper les données de concentration pour chaque commune dans l'île-de-France (Cela a pris approx 100 min)
data_concentration = []

for communes in range(ile_de_fr_commune.shape[0]):
    for particule in particles:
        for year in years:
            concentration = extract_values(url_base, ile_de_fr_commune["Longitude"].loc[communes], ile_de_fr_commune["Latitude"].loc[communes], particule, year)
            dict_year_particle = {"Year": year, "Particle": particule, "Commune": ile_de_fr_commune["Nom_Commune"].loc[communes], "Code_INSEE": ile_de_fr_commune["INSEE_Commune"].loc[communes], "Departement": ile_de_fr_commune["INSEE_Departement"].loc[communes],"Concentration": concentration}
            data_concentration.append(dict_year_particle)


In [ ]:
# Construction du Panel (Commune, année, Partibule_polluante) pour l'analyse
Data_Emissions = pd.DataFrame(data_concentration).pivot_table(index=["Commune", "Code_INSEE", "Departement", "Year"], columns= "Particle", values= "Concentration")

In [ ]:
# Stocker les données sur SSP Cloud

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/concentrations_idf.parquet", "wb") as f:
    Data_Emissions.to_parquet(f)


## Données sur la Mortalité

In [ ]:
url_deces = f"https://www.data.gouv.fr/api/1/datasets/r/0b0c0fbc-af9a-4b9d-9949-b171e7e6fe4a"
deces = pd.read_csv(url_deces, sep=';')

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/deces_idf.parquet", "wb") as f:
    deces.to_parquet(f)

In [ ]:
url_pop = f"https://www.data.gouv.fr/api/1/datasets/r/27361ae9-dcab-4312-97a1-48dab5f39218"
r = requests.get(url_pop)

z = zipfile.ZipFile(io.BytesIO(r.content))

#le fichier que nous souhaitons avoir est celui qui ne contient pas les metadata
fichier_data_pop = [n for n in z.namelist() if "metadata" not in n][0]

#on ouvre le zip
with z.open(fichier_data_pop) as f:
    pop = pd.read_csv(f, sep=";", dtype=str)

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/pop_idf.parquet", "wb") as f:
    pop.to_parquet(f)


## Données sur le prix de l'Immobilier

In [ ]:
url = 'https://static.data.gouv.fr/resources/prix-immobilier-par-commune-ventes-2014-a-2025-dvf-millesime-2026/20261026-112727/prix-immobilier-communes-series-2014-2025.csv'
response = requests.get(url)
df_immo = pd.read_csv(io.BytesIO(response.content))

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/prix_immo.parquet", "wb") as f:
    df_immo.to_parquet(f)

In [ ]:
url = "https://paris.notaires.fr/sites/default/files/HistoriquedesprixaumappartementsanciensParispararrdt_2.pdf"
response = requests.get(url)
response.raise_for_status()

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "tomasmossel"

with fs.open(f"{MY_BUCKET}/diffusion/prix_paris_notaires.pdf", "wb") as f:
    f.write(response.content)